In [1]:
# Code assits by AI(comments, data-preprocessing, visualizations)
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from statsmodels.tsa.stattools import adfuller, coint
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
input_size = 3
output_size = 5
hidden_size = 5
num_layers = 2

In [19]:
"""
nn.Linear is the calculation of this : f(x) = xW^T + bias, where W is weight Matrix.
That is, nn.Linear is combination of (calculation, (Weight matrix + bias vector)).
__init__() method initialize the shape/length of Weight matrix, bias vector
forward() method do the calculation(for nn.Linear case, forward() method of nn.Linear do matrix multiplication and addition)

And this logic applied to all other nn.model_name in pytorch(to be precise, same logic applied to the class that inherit nn.Module).

Docs : https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html
"""
linear = nn.Linear(input_size, output_size)

In [25]:
print(linear.weight.size()) # usually in pytorch, weight matrix W is transposed.
print(linear.bias.size())

torch.Size([5, 3])
torch.Size([5])


In [5]:
lin_params = dict(linear.named_parameters())

In [46]:
print(lin_params)
print('\n')
print('='*70)
print('\n')
print('Weight : ')
print(lin_params['weight'])
print('\n')
print('Bias : ')
print(lin_params['bias'])

{'weight': Parameter containing:
tensor([[-0.2129,  0.3580,  0.3545],
        [-0.3342,  0.0498, -0.3455],
        [-0.2851,  0.5420,  0.0529],
        [-0.2186, -0.1177,  0.5618],
        [-0.4095, -0.4239,  0.3008]], requires_grad=True), 'bias': Parameter containing:
tensor([ 0.4791, -0.1276,  0.0779,  0.0776, -0.2491], requires_grad=True)}




Weight : 
Parameter containing:
tensor([[-0.2129,  0.3580,  0.3545],
        [-0.3342,  0.0498, -0.3455],
        [-0.2851,  0.5420,  0.0529],
        [-0.2186, -0.1177,  0.5618],
        [-0.4095, -0.4239,  0.3008]], requires_grad=True)


Bias : 
Parameter containing:
tensor([ 0.4791, -0.1276,  0.0779,  0.0776, -0.2491], requires_grad=True)


In [23]:
"""
However, even nn.Tanh and nn.ReLU(and others that used as activation functions) is inheriting nn.Module, and act as layer,
since they are not learnable models(like nn.Linear), they contain no weights(just have calculation functionality).

In fact, they are same as F.tanh, F.relu. But why there is nn.Tanh and nn.ReLU? To represent those activation functions as layer!
For example, when we build model using nn.Seqeuntial, we do as below :

# Since all component is 'nn.Module', we can connect those layers sequentially.
model = nn.Sequential(
    nn.Linear(10, 20), # have weights
    nn.ReLU(),         # have no weights (but act as layer)
    nn.Linear(20, 5),  # have weights
    nn.Tanh()          # have no weights (but act as layer)
)

And we usually call these kinds of layers(nn.Tanh, nn.ReLU, etc) as activation function layer
"""
tanh = nn.Tanh()
print(tanh.named_parameters())
print(dict(tanh.named_parameters())) # since they are not learnable models, they named_parameters() will ouput nothing.

<generator object Module.named_parameters at 0x7e566aa42640>
{}


In [7]:
rnn = nn.RNN(input_size, hidden_size, num_layers)
params = dict(rnn.named_parameters())

In [9]:
for param in params.keys():
  if int(param[-1]) == 0:
    print(f'This is {int(param[-1])} Layer')
  else :
    print(f'This is {int(param[-1])} Layer')
  print(param)
  print(params[param])
  print('='*70)

this is 0 Layer
weight_ih_l0
Parameter containing:
tensor([[-0.4435,  0.2812,  0.2756],
        [-0.2629,  0.0399, -0.1176],
        [ 0.3230, -0.1370,  0.1986],
        [ 0.1745, -0.2349, -0.2867],
        [-0.4240,  0.1797,  0.3108]], requires_grad=True)
this is 0 Layer
weight_hh_l0
Parameter containing:
tensor([[ 0.3966,  0.1642, -0.3207, -0.1089,  0.2660],
        [-0.2016,  0.4295, -0.4420,  0.3671,  0.1484],
        [-0.2996,  0.3812, -0.1223,  0.0911, -0.0181],
        [ 0.2831, -0.1783, -0.3746, -0.3675, -0.0035],
        [-0.1277, -0.4179, -0.3560,  0.0920, -0.4303]], requires_grad=True)
this is 0 Layer
bias_ih_l0
Parameter containing:
tensor([ 0.4007,  0.2892,  0.0184, -0.1791, -0.3042], requires_grad=True)
this is 0 Layer
bias_hh_l0
Parameter containing:
tensor([-0.3079, -0.3946,  0.0283,  0.0006,  0.3992], requires_grad=True)
this is 1 Layer
weight_ih_l1
Parameter containing:
tensor([[ 0.3447, -0.2337, -0.3413, -0.0418,  0.0031],
        [ 0.2363, -0.3271,  0.0543, -0.0368,

In [27]:
"""
Time Series Forecasting Strategies: Direct vs. Recursive (generated by Gemini-chat)

This model implements the 'Direct Multi-step Forecasting' strategy.
Below is a comparison to understand why this approach was chosen over the recursive method.

1. Direct Multi-step Forecasting
   ------------------------------------------------------
   - Concept: The model predicts the entire forecast horizon (e.g., next 5 days)
     simultaneously in a single pass ("One-Shot").
   - Mechanism: The final Linear layer maps the hidden state directly to
     an output vector of size N (e.g., output_size=5).
   - Pros:
     1. Stability: Avoids "Error Accumulation" because it doesn't rely on its own past predictions.
     2. Efficiency: Faster inference without the need for a loop.
   - Cons: May capture less of the immediate dependencies between the predicted future steps
     (e.g., how Day 21 affects Day 22).

2. Recursive (Autoregressive) Forecasting (Alternative)
   ----------------------------------------------------
   - Concept: The model predicts one step ahead, appends this prediction to the input,
     and uses it to predict the next step (like writing a story sentence by sentence).
   - Mechanism: Output size is 1. Requires a loop to generate multi-step forecasts.
   - Pros: Logically captures the sequential dependency of the time series.
   - Cons: High risk of "Error Accumulation". If the first prediction (Day 21) is slightly wrong,
     this error propagates to the inputs for Day 22, 23, etc., causing the forecast
     to diverge significantly over time.

Summary:
   For short-term forecasting (e.g., 5~10 days), the Direct method is generally preferred
   to ensure stability and prevent the compounding errors typical of recursive approaches.
"""
class RNNLSTM(nn.Module): # this is direct multi step forcasting model
  def __init__(self, input_size, hidden_size, n_layers, output_size, batch_first=True):
    super(RNNLSTM, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, n_layers, batch_first=batch_first)
    self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers, batch_first=batch_first)
    self.fc = self.fc = nn.Linear(hidden_size, output_size) # this layer determine the size(dimension) of output vector

  def forward(self, x):
    out, _ = self.rnn(x)
    out, (hidden, cell) = self.lstm(out)

    # if we need all the output from every time step, then
    # out = self.fc(out)

    # but this time, we only need the output from last time step of LSTM.
    last_out = out[:, -1, :] # (Batch, Hidden_Size)
    final_out = self.fc(last_out) # (Batch, Output_Size)
    return final_out

"""
## This is Recursive (Autoregressive) Forecasting version with RNNLSTM. (generated by Gemini-chat)
# 1. the model predict right next session(only one next session) (output_size=1)
model = RNNLSTM(input_size=1, hidden_size=64, output_size=1, n_layers=2)

# 2.Inference
current_input = data_last_20_days # (1, 20, 1)
predicted_5_days = []

for i in range(5):
    # A. one day prediction
    next_value = model(current_input) # result : (1, 1) -> tomorrow 주가

    # B. save predicted value
    predicted_5_days.append(next_value.item())

    # C. update the input (drop oldest data, and add prediction(output of the model) value we just predicted)
    # sliding window : [0~19] -> [1~19 + next_value]
    next_value_reshaped = next_value.unsqueeze(1) # align dimension to (1, 1, 1)
    current_input = torch.cat((current_input[:, 1:, :], next_value_reshaped), dim=1)

print(predicted_5_days)
"""

In [29]:
# let's assume we are doing stock price forcasting.
input_size = 3 # look at last 3 days
hidden_size = 5
n_layers = 2
output_size = 3 # it means we are forcasting stock price for next 3 days.
rnnlstm = RNNLSTM(input_size, hidden_size, n_layers, output_size)

In [34]:
print(dict(rnnlstm.named_modules())) # it has 2 layer for each models
print('\n')
print('='*70)
print('\n')
print('Number of layer in RNNLSTM model is ', len(dict(rnnlstm.named_modules()))) # it has 2 layer for each models

{'': RNNLSTM(
  (rnn): RNN(3, 5, num_layers=2, batch_first=True)
  (lstm): LSTM(5, 5, num_layers=2, batch_first=True)
  (fc): Linear(in_features=5, out_features=3, bias=True)
), 'rnn': RNN(3, 5, num_layers=2, batch_first=True), 'lstm': LSTM(5, 5, num_layers=2, batch_first=True), 'fc': Linear(in_features=5, out_features=3, bias=True)}




Number of layer in RNNLSTM model is  4


In [ ]:
"""
Where exactly does the code catch and consume return value of forward() method? (generated by Gemini-chat)
The answer is: It is captured by a variable inside our "Training Loop" (the main code that runs the learning process).
Let's trace the return value in the most common scenario: Model Training.
The value returned by forward is typically caught by a variable named prediction (or output).
This variable is then immediately fed into the Loss Function to be compared against the correct answer (ground truth).
Look at the code below, and you will see exactly where the hand-off happens:

# ======== CODE ========
# 1. Preparation
model = RNNLSTM(...)
criterion = nn.MSELoss() # The Judge (Calculates error)

# --- This is the "Main Code (Training Loop)" ---
for input_data, true_answer in dataloader:

    # [1] Catches the return value HERE! (The Receiver)
    # Calling 'model(input_data)' triggers 'forward',
    # and the RETURN value is assigned to the 'prediction' variable.
    prediction = model(input_data)

    # [2] How is it used? (The Usage)
    # "Hey, how different is this 'prediction' from the 'true_answer'?"
    # The 'prediction' variable is consumed here by the loss function.
    loss = criterion(prediction, true_answer)

    # [3] Learn from the error (Backpropagation)
    loss.backward()
    optimizer.step()
"""

"""
What about during Real-World Use (Inference)?
When training is finished and you are using the model to predict actual stock prices, You (the user) or your Application are the final consumer.

# ======== CODE ========
# Real-world Inference
forecast = model(today_data) # The 'forecast' variable catches the return value.

print(forecast) # Consumed by your eyes (via print)
# OR
buy_stock(forecast) # Consumed by a trading function (logic)
"""